In [ ]:
  # ==================== 支持手动指定Invasive的多fold数据集生成 ====================
import os
import openslide
import numpy as np
from PIL import Image, ImageEnhance
import cv2
import random
from tqdm import tqdm
import concurrent.futures
from collections import Counter, defaultdict
import pickle

# ==================== 配置参数 ====================
slide_labels = {
    "01.svs": "Normal", "02.svs": "Benign", "03.svs": "In_Situ", "04.svs": "Normal",
    "05.svs": "Benign", "06.svs": "Invasive", "07.svs": "In_Situ", "08.svs": "Invasive",
    "09.svs": "Benign", "10.svs": "Normal", "11.svs": "In_Situ", "12.svs": "Invasive",
    "13.svs": "Invasive", "14.svs": "Benign", "15.svs": "In_Situ", "16.svs": "Invasive",
    "17.svs": "Normal", "18.svs": "In_Situ", "19.svs": "Invasive", "20.svs": "Benign",
    "A01.svs": "Invasive", "A02.svs": "Invasive", "A03.svs": "Invasive", "A04.svs": "Invasive",
    "A05.svs": "Invasive", "A06.svs": "Invasive", "A07.svs": "Invasive", "A08.svs": "Invasive",
    "A09.svs": "Invasive", "A10.svs": "Invasive"
}

WSI_DIR = "./WSI"
OUTPUT_ROOT = "./wsi_patches_multi_fold_voting"
AUG_DIR = os.path.join(WSI_DIR, "augmented")
os.makedirs(AUG_DIR, exist_ok=True)
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# 切片参数
PATCH_SIZE = 224
LEVEL = 0
MAX_PER_WSI = 3000
TISSUE_THRESH_TRAIN = 0.30
TISSUE_THRESH_VAL = 0.70

# 均衡目标
TARGET_CLASS_COUNT = 6  # 每类6个WSI（完全均衡）

# ==================== 手动指定的Invasive选择（可按需扩展） ====================
MANUAL_INVASIVE_SELECTIONS = {
    # Fold 1: 主要用原始编号的Invasive
    1: ['06.svs', '08.svs', '12.svs', '13.svs', '16.svs', '19.svs'],
    
    # Fold 2: 主要用A系列的Invasive
    2: ['A01.svs', 'A02.svs', 'A03.svs', 'A04.svs', 'A05.svs', 'A06.svs'],
    
    # Fold 3: 混合但偏向后期
    3: ['A07.svs', 'A08.svs', 'A09.svs', 'A10.svs', '12.svs', '19.svs'],
    
    # Fold 4: 均匀混合
    4: ['06.svs', '13.svs', 'A01.svs', 'A05.svs', 'A08.svs', 'A10.svs'],
    
    # 可以继续添加更多fold...
    # 5: ['08.svs', '16.svs', 'A02.svs', 'A06.svs', 'A09.svs', '19.svs'],
    # 6: ['12.svs', 'A03.svs', 'A04.svs', 'A07.svs', '13.svs', '06.svs'],
}

# ==================== 核心修改：支持手动选择Invasive的均衡函数 ====================
def balance_wsis_with_manual_selection(train_wsi_files, fold_idx, invasive_selection=None):
    """
    支持手动指定Invasive选择的均衡化
    fold_idx: fold编号（1开始）
    invasive_selection: 手动指定的Invasive WSI列表，如果为None则自动选择
    """
    print(f"\n📊 WSI层面均衡化 (Fold {fold_idx})...")
    
    # 统计原始分布
    class_distribution = Counter([slide_labels[w] for w in train_wsi_files])
    print(f"原始分布: {dict(class_distribution)}")
    print(f"目标每类: {TARGET_CLASS_COUNT}个WSI")
    
    # ==================== 关键修改：手动处理Invasive选择 ====================
    # 获取该fold的手动选择（如果提供了）
    if invasive_selection is None:
        # 默认使用预设的选择
        invasive_selection = MANUAL_INVASIVE_SELECTIONS.get(fold_idx, None)
    
    if invasive_selection:
        print(f"\n🎯 Fold {fold_idx} 手动指定的Invasive WSI:")
        for wsi in invasive_selection:
            print(f"  - {wsi}")
        
        # 验证选择的WSI都在训练集中
        valid_selection = [w for w in invasive_selection if w in train_wsi_files]
        if len(valid_selection) < TARGET_CLASS_COUNT:
            print(f"⚠️  警告：只有{len(valid_selection)}个选择的WSI在训练集中")
            # 补充剩余名额
            remaining = [w for w in train_wsi_files 
                        if slide_labels[w] == 'Invasive' and w not in valid_selection]
            needed = TARGET_CLASS_COUNT - len(valid_selection)
            if needed > 0 and remaining:
                additional = random.sample(remaining, min(needed, len(remaining)))
                valid_selection.extend(additional)
                print(f"  补充了{len(additional)}个Invasive WSI: {additional}")
        
        # 限制数量为目标值
        selected_invasive = valid_selection[:TARGET_CLASS_COUNT]
        print(f"✅ 最终选择{len(selected_invasive)}个Invasive WSI用于本fold")
    else:
        # 自动选择：随机选择TARGET_CLASS_COUNT个Invasive
        invasive_wsis = [w for w in train_wsi_files if slide_labels[w] == 'Invasive']
        selected_invasive = random.sample(invasive_wsis, 
                                         min(TARGET_CLASS_COUNT, len(invasive_wsis)))
        print(f"🎲 自动随机选择Invasive WSI: {selected_invasive}")
    
    # ==================== 处理其他类别增强 ====================
    augmented_wsi_list = []
    augmentation_records = []
    
    for cls in ['Normal', 'Benign', 'In_Situ']:
        current_count = class_distribution.get(cls, 0)
        needed = TARGET_CLASS_COUNT - current_count
        
        if needed > 0:
            print(f"\n  🎯 类别 {cls}:")
            print(f"     当前: {current_count}张")
            print(f"     需要增强: {needed}张")
            
            # 获取该类所有WSI
            cls_wsis = [w for w in train_wsi_files if slide_labels[w] == cls]
            
            if not cls_wsis:
                print(f"     ⚠️  没有基础WSI可用于增强")
                continue
            
            for i in range(needed):
                # 随机选择一个基础WSI
                base_wsi = random.choice(cls_wsis)
                base_path = os.path.join(WSI_DIR, base_wsi)
                
                # 生成增强文件名（包含fold信息）
                aug_id = f"{i:03d}"
                aug_name = f"AUG_{cls}_F{fold_idx}_{aug_id}_{base_wsi.replace('.svs', '')}.svs"
                
                print(f"     生成: {aug_name} (基于 {base_wsi})")
                
                # 创建增强缩略图（使用你原来的函数）
                thumb, meta = create_augmented_wsi_thumbnail(
                    base_path, 
                    aug_name.replace('.svs', ''),
                    augmentation_type=None
                )
                
                if thumb is not None:
                    augmented_wsi_list.append({
                        'file': aug_name,
                        'label': cls,
                        'is_augmented': True,
                        'original': base_wsi,
                        'aug_meta': meta,
                        'fold': fold_idx
                    })
                    
                    augmentation_records.append({
                        'aug_wsi': aug_name,
                        'original': base_wsi,
                        'class': cls,
                        'fold': fold_idx,
                        'methods': meta['augmentation_type']
                    })
    
    # ==================== 构建最终的WSI列表 ====================
    all_wsis = []
    
    # 1. 添加非Invasive的原始WSI
    for wsi in train_wsi_files:
        cls = slide_labels[wsi]
        if cls != 'Invasive':
            all_wsis.append({
                'file': wsi,
                'label': cls,
                'is_augmented': False,
                'original': wsi,
                'fold': fold_idx
            })
    
    # 2. 添加手动选择的Invasive WSI
    for wsi in selected_invasive:
        all_wsis.append({
            'file': wsi,
            'label': 'Invasive',
            'is_augmented': False,
            'original': wsi,
            'fold': fold_idx,
            'manually_selected': True
        })
    
    # 3. 添加增强WSI
    all_wsis.extend(augmented_wsi_list)
    
    # 验证均衡性
    final_dist = Counter([w['label'] for w in all_wsis])
    print(f"\n✅ Fold {fold_idx} 均衡化完成!")
    print(f"   最终WSI分布: {dict(final_dist)}")
    print(f"   Invasive选择: {selected_invasive}")
    print(f"   总WSI数: {len(all_wsis)} (原始: {len(train_wsi_files)}, 增强: {len(augmented_wsi_list)})")
    
    return all_wsis, augmentation_records, selected_invasive

# ==================== 修改主处理流程 ====================
def process_fold_with_manual_selection(fold_idx, train_wsi_files, val_wsi_files, 
                                      custom_invasive_selection=None):
    """
    处理单个fold，支持手动选择Invasive
    custom_invasive_selection: 可覆盖预设的手动选择
    """
    print(f"\n{'='*60}")
    print(f"🎯 处理 Fold {fold_idx} (支持手动选择)")
    print(f"{'='*60}")
    
    # 阶段1: WSI层面均衡化（支持手动选择）
    print("\n📈 阶段1: WSI均衡化（手动选择Invasive）")
    balanced_train_wsis, aug_records, invasive_selected = balance_wsis_with_manual_selection(
        train_wsi_files, fold_idx, custom_invasive_selection
    )
    
    # 保存选择记录
    selection_record = {
        'fold': fold_idx,
        'invasive_selected': invasive_selected,
        'invasive_available': [w for w in train_wsi_files if slide_labels[w] == 'Invasive'],
        'augmentation_count': len(aug_records)
    }
    
    # 阶段2: 并行切片处理（使用你原来的process_wsi_to_patches函数）
    print("\n✂️ 阶段2: 批量切片处理")
    
    # 准备所有任务
    all_tasks = []
    
    # 训练集任务
    for wsi_info in balanced_train_wsis:
        all_tasks.append((wsi_info, "train", fold_idx-1))  # fold_idx-1因为原来的函数从0开始
    
    # 验证集任务
    for wsi_file in val_wsi_files:
        wsi_info = {
            'file': wsi_file,
            'label': slide_labels[wsi_file],
            'is_augmented': False,
            'original': wsi_file,
            'fold': fold_idx
        }
        all_tasks.append((wsi_info, "val", fold_idx-1))
    
    print(f"  总任务数: {len(all_tasks)}")
    print(f"  训练WSI: {len(balanced_train_wsis)} (含增强)")
    print(f"  验证WSI: {len(val_wsi_files)}")
    print(f"  选择的Invasive: {invasive_selected}")
    
    # 并行处理（使用你原来的切片函数）
    results = []
    with concurrent.futures.ProcessPoolExecutor(max_workers=8) as executor:
        futures = [executor.submit(process_wsi_to_patches, *task) for task in all_tasks]
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
            results.append(future.result())
    
    # 统计结果
    total_patches = sum(r[1] for r in results if r is not None)
    print(f"\n✅ Fold {fold_idx} 完成!")
    print(f"   生成切片总数: {total_patches}")
    
    return {
        'fold': fold_idx,
        'original_train_wsis': train_wsi_files,
        'balanced_train_wsis': [w['file'] for w in balanced_train_wsis],
        'val_wsis': val_wsi_files,
        'invasive_selected': invasive_selected,
        'augmentation_records': aug_records,
        'selection_record': selection_record,
        'patch_count': total_patches
    }

# ==================== 扩展的多fold生成主函数 ====================
def generate_multi_fold_dataset(num_folds=4, start_fold=1):
    """
    生成支持多fold扩展的数据集
    num_folds: 要生成的fold数量
    start_fold: 起始fold编号
    """
    print(f"🚀 开始生成多fold数据集 (Fold {start_fold} 到 {start_fold+num_folds-1})")
    print("="*60)
    
    wsi_files = list(slide_labels.keys())
    
    # 定义基础4-fold划分
    base_val_sets = [
        # Fold 1
        ["01.svs", "04.svs", "02.svs", "05.svs", 
         "03.svs", "07.svs", "06.svs", "12.svs"],
        # Fold 2  
        ["10.svs", "17.svs", "09.svs", "14.svs",
         "11.svs", "15.svs", "08.svs", "13.svs"],
        # Fold 3
        ["01.svs", "10.svs", "20.svs", "05.svs",
         "18.svs", "07.svs", "A01.svs", "A02.svs"],
        # Fold 4
        ["04.svs", "17.svs", "02.svs", "09.svs",
         "03.svs", "11.svs", "A03.svs", "A04.svs"]
    ]
    
    # 如果需要的fold数超过基础划分，可以循环使用或生成新划分
    if num_folds > len(base_val_sets):
        print(f"⚠️  需要{num_folds}个fold，但只有{len(base_val_sets)}个基础划分")
        print("  将循环使用基础划分...")
        # 循环使用基础划分
        val_sets = []
        for i in range(num_folds):
            val_sets.append(base_val_sets[i % len(base_val_sets)])
    else:
        val_sets = base_val_sets[:num_folds]
    
    # 处理每个fold
    all_results = []
    all_selection_records = []
    
    for i in range(num_folds):
        fold_idx = start_fold + i
        val_wsis = val_sets[i]
        train_wsis = [w for w in wsi_files if w not in val_wsis]
        
        print(f"\n{'='*60}")
        print(f"处理 Fold {fold_idx}/{start_fold+num_folds-1}")
        print(f"{'='*60}")
        
        # 处理当前fold
        fold_result = process_fold_with_manual_selection(fold_idx, train_wsis, val_wsis)
        all_results.append(fold_result)
        all_selection_records.append(fold_result['selection_record'])
    
    # ==================== 保存详细的元数据 ====================
    print(f"\n{'='*60}")
    print("📦 保存数据集元数据")
    print(f"{'='*60}")
    
    # 1. 保存总体结果
    summary_path = os.path.join(OUTPUT_ROOT, f"multi_fold_summary_f{start_fold}_to_{start_fold+num_folds-1}.pkl")
    with open(summary_path, 'wb') as f:
        pickle.dump({
            'all_results': all_results,
            'num_folds': num_folds,
            'start_fold': start_fold,
            'target_class_count': TARGET_CLASS_COUNT,
            'manual_selections': MANUAL_INVASIVE_SELECTIONS
        }, f)
    
    # 2. 保存选择记录（便于分析）
    selection_path = os.path.join(OUTPUT_ROOT, f"invasive_selection_records.csv")
    with open(selection_path, 'w') as f:
        f.write("fold,selected_invasive,available_invasive,augmentation_count\n")
        for record in all_selection_records:
            selected = ';'.join(record['invasive_selected'])
            available = ';'.join(record['invasive_available'])
            f.write(f"{record['fold']},{selected},{available},{record['augmentation_count']}\n")
    
    # 3. 生成训练集成用的配置文件
    config_path = os.path.join(OUTPUT_ROOT, f"ensemble_config.json")
    config = {
        "num_models": num_folds,
        "model_paths": [f"fold{start_fold+i}" for i in range(num_folds)],
        "invasive_selections": {r['fold']: r['invasive_selected'] for r in all_selection_records},
        "voting_method": "soft_voting",  # 或 "hard_voting"
        "weights": [1.0] * num_folds  # 可以设置不同权重
    }
    
    import json
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    
    # ==================== 打印统计报告 ====================
    print(f"\n{'='*60}")
    print("🎉 多fold数据集生成完成！")
    print(f"{'='*60}")
    
    total_original = sum(len(r['original_train_wsis']) for r in all_results)
    total_balanced = sum(len(r['balanced_train_wsis']) for r in all_results)
    total_augmented = total_balanced - total_original
    total_patches = sum(r['patch_count'] for r in all_results)
    
    print(f"📊 总体统计:")
    print(f"  生成的fold数: {num_folds}")
    print(f"  原始训练WSI总数: {total_original}")
    print(f"  增强WSI总数: {total_augmented}")
    print(f"  均衡后WSI总数: {total_balanced}")
    print(f"  总切片数: {total_patches}")
    
    print(f"\n📊 Invasive选择多样性分析:")
    all_selected = []
    for record in all_selection_records:
        all_selected.extend(record['invasive_selected'])
    
    selection_counts = Counter(all_selected)
    print(f"  共使用了{len(selection_counts)}个不同的Invasive WSI")
    print(f"  使用频率分布:")
    for wsi, count in selection_counts.most_common():
        print(f"    {wsi}: {count}个fold使用")
    
    print(f"\n🎯 集成学习优势:")
    print(f"  1. 不同fold看到不同的Invasive WSI组合")
    print(f"  2. 模型学习到更全面的Invasive特征")
    print(f"  3. 集成投票提高预测鲁棒性")
    print(f"  4. 方便后续扩展更多fold")
    
    print(f"\n📁 生成的文件:")
    print(f"  数据集目录: {OUTPUT_ROOT}")
    print(f"  元数据: {summary_path}")
    print(f"  选择记录: {selection_path}")
    print(f"  集成配置: {config_path}")
    print(f"{'='*60}")

# ==================== 使用示例 ====================
if __name__ == "__main__":
    # 生成4个fold的数据集
    generate_multi_fold_dataset(num_folds=4, start_fold=1)
    
    # 后续可以轻松扩展：
    # generate_multi_fold_dataset(num_folds=2, start_fold=5)  # 生成fold5-6
    # generate_multi_fold_dataset(num_folds=2, start_fold=7)  # 生成fold7-8

🚀 开始生成多fold数据集 (Fold 1 到 4)

处理 Fold 1/4

🎯 处理 Fold 1 (支持手动选择)

📈 阶段1: WSI均衡化（手动选择Invasive）

📊 WSI层面均衡化 (Fold 1)...
原始分布: {'Invasive': 14, 'Benign': 3, 'Normal': 2, 'In_Situ': 3}
目标每类: 6个WSI

🎯 Fold 1 手动指定的Invasive WSI:
  - 06.svs
  - 08.svs
  - 12.svs
  - 13.svs
  - 16.svs
  - 19.svs
⚠️  警告：只有4个选择的WSI在训练集中
  补充了2个Invasive WSI: ['A03.svs', 'A02.svs']
✅ 最终选择6个Invasive WSI用于本fold

  🎯 类别 Normal:
     当前: 2张
     需要增强: 4张
     生成: AUG_Normal_F1_000_10.svs (基于 10.svs)
     生成: AUG_Normal_F1_001_17.svs (基于 17.svs)
     生成: AUG_Normal_F1_002_10.svs (基于 10.svs)
     生成: AUG_Normal_F1_003_10.svs (基于 10.svs)

  🎯 类别 Benign:
     当前: 3张
     需要增强: 3张
     生成: AUG_Benign_F1_000_20.svs (基于 20.svs)
     生成: AUG_Benign_F1_001_20.svs (基于 20.svs)
     生成: AUG_Benign_F1_002_09.svs (基于 09.svs)

  🎯 类别 In_Situ:
     当前: 3张
     需要增强: 3张
     生成: AUG_In_Situ_F1_000_18.svs (基于 18.svs)
     生成: AUG_In_Situ_F1_001_18.svs (基于 18.svs)
     生成: AUG_In_Situ_F1_002_15.svs (基于 15.svs)

✅ Fold 1 均衡化完成!
   最终WSI分布: 

  0%|          | 0/32 [00:00<?, ?it/s]

  Processing: 09.svs (Benign)  Processing: 15.svs (In_Situ)  Processing: 14.svs (Benign)  Processing: 11.svs (In_Situ)

  Processing: 10.svs (Normal)  Processing: 18.svs (In_Situ)  Processing: 20.svs (Benign)
  Processing: 17.svs (Normal)




    ✅ 保存了 3000 个切片


  3%|▎         | 1/32 [02:00<1:02:23, 120.76s/it]

  Processing: 08.svs (Invasive)
    ✅ 保存了 3000 个切片


...

    ✅ 保存了 3000 个切片


 97%|█████████▋| 31/32 [13:31<00:24, 24.61s/it]

    ✅ 保存了 3000 个切片


100%|██████████| 32/32 [14:12<00:00, 26.64s/it]


✅ Fold 4 完成!
   生成切片总数: 96000

📦 保存数据集元数据

🎉 多fold数据集生成完成！
📊 总体统计:
  生成的fold数: 4
  原始训练WSI总数: 88
  增强WSI总数: 8
  均衡后WSI总数: 96
  总切片数: 384000

📊 Invasive选择多样性分析:
  共使用了16个不同的Invasive WSI
  使用频率分布:
    13.svs: 2个fold使用
    19.svs: 2个fold使用
    A03.svs: 2个fold使用
    A02.svs: 2个fold使用
    A01.svs: 2个fold使用
    A05.svs: 2个fold使用
    A08.svs: 2个fold使用
    A10.svs: 2个fold使用
    08.svs: 1个fold使用
    16.svs: 1个fold使用
    A04.svs: 1个fold使用
    A06.svs: 1个fold使用
    A07.svs: 1个fold使用
    A09.svs: 1个fold使用
    12.svs: 1个fold使用
    06.svs: 1个fold使用

🎯 集成学习优势:
  1. 不同fold看到不同的Invasive WSI组合
  2. 模型学习到更全面的Invasive特征
  3. 集成投票提高预测鲁棒性
  4. 方便后续扩展更多fold

📁 生成的文件:
  数据集目录: ./wsi_patches_multi_fold_voting
  元数据: ./wsi_patches_multi_fold_voting/multi_fold_summary_f1_to_4.pkl
  选择记录: ./wsi_patches_multi_fold_voting/invasive_selection_records.csv
  集成配置: ./wsi_patches_multi_fold_voting/ensemble_config.json


In [11]:
# ==================== 完整修复版检验程序 ====================
import os
import pickle
import numpy as np
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
from tqdm import tqdm

print("🔍 开始按fold独立检验数据集质量（修复版）...")
print("="*80)

# ==================== 配置参数 ====================
DATASET_ROOT = "./wsi_patches_multi_fold_voting"
AUG_DIR = "./WSI/augmented"

# 原始标签定义
slide_labels = {
    "01.svs": "Normal", "02.svs": "Benign", "03.svs": "In_Situ", "04.svs": "Normal",
    "05.svs": "Benign", "06.svs": "Invasive", "07.svs": "In_Situ", "08.svs": "Invasive",
    "09.svs": "Benign", "10.svs": "Normal", "11.svs": "In_Situ", "12.svs": "Invasive",
    "13.svs": "Invasive", "14.svs": "Benign", "15.svs": "In_Situ", "16.svs": "Invasive",
    "17.svs": "Normal", "18.svs": "In_Situ", "19.svs": "Invasive", "20.svs": "Benign",
    "A01.svs": "Invasive", "A02.svs": "Invasive", "A03.svs": "Invasive", "A04.svs": "Invasive",
    "A05.svs": "Invasive", "A06.svs": "Invasive", "A07.svs": "Invasive", "A08.svs": "Invasive",
    "A09.svs": "Invasive", "A10.svs": "Invasive"
}

# ==================== 修复的解析函数 ====================
def get_wsi_source_from_slice_fixed(slice_filename):
    """
    修复版：从切片文件名正确获取WSI来源
    切片: AUG_Normal_002_10_01543.jpg → WSI: AUG_Normal_002_10.svs
    """
    if slice_filename.startswith('AUG_') and slice_filename.endswith('.jpg'):
        # 解析增强切片
        basename = slice_filename[:-4]  # 去掉.jpg
        parts = basename.split('_')
        
        # 格式: AUG_Class_ID_Original_Index
        if len(parts) >= 5:
            # 类别可能是多部分的（如In_Situ）
            possible_classes = ['Normal', 'Benign', 'In_Situ', 'Invasive']
            
            # 查找类别
            aug_class = None
            for i in range(2, 4):  # 类别可能由1-3部分组成
                test_class = '_'.join(parts[1:1+i])
                if test_class in possible_classes:
                    aug_class = test_class
                    remaining = parts[1+i:]
                    break
            
            if not aug_class:
                # 如果没有匹配，使用简单解析
                aug_class = parts[1]
                remaining = parts[2:]
            
            # 提取ID和原始WSI
            if len(remaining) >= 2:
                aug_id = remaining[0]
                
                # 原始WSI是ID后的部分（去掉最后的Index）
                original_parts = remaining[1:-1]  # 去掉最后的Index
                original_base = '_'.join(original_parts)
                original_wsi = f"{original_base}.svs"
                
                # 构建增强WSI名称
                wsi_source = f"AUG_{aug_class}_{aug_id}_{original_base}.svs"
                return wsi_source, True, original_wsi
    else:
        # 原始切片
        wsi_name = slice_filename.split('_')[0] + '.svs'
        return wsi_name, False, wsi_name
    
    return None, False, None

def parse_augmented_wsi_name(wsi_filename):
    """解析增强WSI文件名"""
    if not wsi_filename.startswith('AUG_'):
        return None, None, None
    
    basename = wsi_filename.replace('.svs', '')
    parts = basename.split('_')
    
    if len(parts) < 4:
        return None, None, None
    
    # 处理类别
    possible_classes = ['Normal', 'Benign', 'In_Situ', 'Invasive']
    aug_class = None
    
    for i in range(2, 4):
        test_class = '_'.join(parts[1:1+i])
        if test_class in possible_classes:
            aug_class = test_class
            remaining = parts[1+i:]
            break
    
    if not aug_class:
        aug_class = parts[1]
        remaining = parts[2:]
    
    # 提取ID和原始WSI
    if len(remaining) >= 1:
        aug_id = remaining[0]
        original_parts = remaining[1:] if len(remaining) > 1 else []
        original_base = '_'.join(original_parts) if original_parts else ''
        original_wsi = f"{original_base}.svs" if original_base else ''
        
        return aug_class, aug_id, original_wsi
    
    return None, None, None

# ==================== 修复的检验函数 ====================
def check_single_fold_fixed(fold_num):
    """修复版：独立检验单个fold的数据集"""
    print(f"\n{'='*60}")
    print(f"🔍 检验 Fold {fold_num}（修复版）")
    print(f"{'='*60}")
    
    fold_dir = os.path.join(DATASET_ROOT, f"fold{fold_num}")
    
    if not os.path.exists(fold_dir):
        print(f"❌ Fold {fold_num} 目录不存在")
        return [], []
    
    issues = []
    stats = {
        'train_slices': 0,
        'val_slices': 0,
        'train_wsi_sources': set(),
        'val_wsi_sources': set(),
        'train_wsi_counts': defaultdict(int),  # WSI级别的统计
        'val_wsi_counts': defaultdict(int),
        'class_distribution': defaultdict(lambda: {'train': 0, 'val': 0}),
        'augmented_slices': {'train': 0, 'val': 0},
        'augmented_wsis': set()  # 增强WSI集合
    }
    
    # ==================== 1. 数据泄露检验 ====================
    print("\n1️⃣ 数据泄露检验 (本fold内)")
    print("-"*40)
    
    # 检查train集
    train_dir = os.path.join(fold_dir, "train")
    if os.path.exists(train_dir):
        for class_name in os.listdir(train_dir):
            class_dir = os.path.join(train_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            
            # 获取所有切片文件
            slice_files = [f for f in os.listdir(class_dir) if f.endswith('.jpg')]
            if not slice_files:
                continue
            
            # 统计切片数量
            stats['train_slices'] += len(slice_files)
            stats['class_distribution'][class_name]['train'] += len(slice_files)
            
            # 抽样检查WSI来源
            sample_size = min(100, len(slice_files))
            for slice_file in slice_files[:sample_size]:
                wsi_source, is_augmented, original_wsi = get_wsi_source_from_slice_fixed(slice_file)
                
                if wsi_source:
                    stats['train_wsi_sources'].add(wsi_source)
                    stats['train_wsi_counts'][wsi_source] += 1
                    
                    if is_augmented:
                        stats['augmented_slices']['train'] += 1
                        stats['augmented_wsis'].add(wsi_source)
                    
                    # 检查类别一致性（修复In_Situ问题）
                    if is_augmented:
                        # 解析增强WSI的类别
                        aug_class, _, _ = parse_augmented_wsi_name(wsi_source)
                        if aug_class and aug_class != class_name:
                            # 只报告一次，避免重复
                            key = f"类别不匹配:{wsi_source}"
                            if key not in issues:
                                issues.append(f"增强WSI类别不匹配: {wsi_source} (目录:{class_name}, 文件:{aug_class})")
                    elif original_wsi in slide_labels:
                        actual_class = slide_labels[original_wsi]
                        if actual_class != class_name:
                            issues.append(f"原始切片类别不匹配: {slice_file} (目录:{class_name}, WSI:{actual_class})")
    
    # 检查val集
    val_dir = os.path.join(fold_dir, "val")
    if os.path.exists(val_dir):
        for class_name in os.listdir(val_dir):
            class_dir = os.path.join(val_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            
            slice_files = [f for f in os.listdir(class_dir) if f.endswith('.jpg')]
            if not slice_files:
                continue
            
            stats['val_slices'] += len(slice_files)
            stats['class_distribution'][class_name]['val'] += len(slice_files)
            
            sample_size = min(50, len(slice_files))
            for slice_file in slice_files[:sample_size]:
                wsi_source, is_augmented, original_wsi = get_wsi_source_from_slice_fixed(slice_file)
                
                if wsi_source:
                    stats['val_wsi_sources'].add(wsi_source)
                    stats['val_wsi_counts'][wsi_source] += 1
                    
                    if is_augmented:
                        stats['augmented_slices']['val'] += 1
                        issues.append(f"验证集不应包含增强数据: {slice_file}")
    
    # 检查本fold内训练集和验证集的WSI重叠
    fold_overlap = stats['train_wsi_sources'] & stats['val_wsi_sources']
    
    if fold_overlap:
        print(f"❌ 本fold内数据泄露: {len(fold_overlap)}个WSI同时出现在训练集和验证集")
        for wsi in list(fold_overlap)[:5]:
            print(f"  - {wsi}")
        issues.append(f"数据泄露: {len(fold_overlap)}个WSI重叠")
    else:
        print(f"✅ 本fold内无数据泄露")
    
    # ==================== 2. 增强数据检验（修复版） ====================
    print(f"\n2️⃣ 增强数据检验（修复版）")
    print("-"*40)
    
    # 检查增强WSI的元数据
    if os.path.exists(AUG_DIR):
        missing_metadata_count = 0
        
        for aug_wsi in stats['augmented_wsis']:
            # 构建元数据文件名
            wsi_base = aug_wsi.replace('.svs', '')
            meta_file = f"{wsi_base}_meta.pkl"
            meta_path = os.path.join(AUG_DIR, meta_file)
            
            if not os.path.exists(meta_path):
                missing_metadata_count += 1
                # 只报告一次
                if f"缺少元数据:{aug_wsi}" not in issues:
                    issues.append(f"增强WSI缺少元数据: {aug_wsi}")
        
        if missing_metadata_count > 0:
            print(f"❌ 增强数据问题: {missing_metadata_count}个增强WSI缺少元数据")
        else:
            print(f"✅ 所有增强WSI都有元数据")
    else:
        print(f"⚠️  增强数据目录不存在")
    
    print(f"✅ 训练集包含增强切片: {stats['augmented_slices']['train']}个")
    
    if stats['augmented_slices']['val'] > 0:
        print(f"❌ 验证集包含增强切片: {stats['augmented_slices']['val']}个 (不应有增强)")
    else:
        print(f"✅ 验证集无增强数据 (正确)")
    
    # ==================== 3. 数据集均衡性检验（WSI层面） ====================
    print(f"\n3️⃣ 数据集均衡性检验（WSI层面）")
    print("-"*40)
    
    print(f"📊 切片数量统计:")
    print(f"  训练集: {stats['train_slices']}个切片")
    print(f"  验证集: {stats['val_slices']}个切片")
    
    print(f"\n📊 WSI来源统计（这才是关键！）:")
    print(f"  训练集WSI: {len(stats['train_wsi_sources'])}个")
    print(f"  验证集WSI: {len(stats['val_wsi_sources'])}个")
    print(f"  其中增强WSI: {len(stats['augmented_wsis'])}个")
    
    # WSI层面的类别分布
    print(f"\n📊 WSI层面类别分布（训练集）:")
    wsi_class_dist = defaultdict(int)
    
    for wsi_source in stats['train_wsi_sources']:
        if wsi_source.startswith('AUG_'):
            # 增强WSI
            aug_class, _, _ = parse_augmented_wsi_name(wsi_source)
            if aug_class:
                wsi_class_dist[aug_class] += 1
        else:
            # 原始WSI
            if wsi_source in slide_labels:
                wsi_class_dist[slide_labels[wsi_source]] += 1
    
    for cls in ['Normal', 'Benign', 'In_Situ', 'Invasive']:
        count = wsi_class_dist.get(cls, 0)
        print(f"  {cls}: {count}个WSI")
    
    # 切片层面的类别分布（仅供参考）
    print(f"\n📊 切片层面类别分布（训练集，仅供参考）:")
    for cls in ['Normal', 'Benign', 'In_Situ', 'Invasive']:
        count = stats['class_distribution'][cls]['train']
        print(f"  {cls}: {count}个切片")
    
    # 检查WSI层面的均衡性（这才是重要的！）
    if wsi_class_dist:
        values = list(wsi_class_dist.values())
        mean_val = np.mean(values)
        std_val = np.std(values)
        cv = std_val / mean_val if mean_val > 0 else 0
        
        print(f"\n📈 WSI层面均衡性指标:")
        print(f"  每类平均WSI数: {mean_val:.1f}")
        print(f"  标准差: {std_val:.1f}")
        print(f"  变异系数(CV): {cv:.3f}")
        
        if cv < 0.3:
            print(f"  ✅ WSI层面均衡性良好")
        elif cv < 0.5:
            print(f"  ⚠️  WSI层面均衡性一般")
        else:
            print(f"  ❌ WSI层面均衡性较差")
            issues.append(f"WSI层面均衡性差: CV={cv:.3f}")
    
    # ==================== 4. 切片-WSI映射检验 ====================
    print(f"\n4️⃣ 切片-WSI映射检验")
    print("-"*40)
    
    mapping_issues = 0
    checked_slices = 0
    
    # 快速检查一些样本
    for split_type in ['train']:  # 只检查训练集
        split_dir = os.path.join(fold_dir, split_type)
        if not os.path.exists(split_dir):
            continue
        
        for class_name in ['Normal', 'Benign', 'In_Situ', 'Invasive']:
            class_dir = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            
            slice_files = [f for f in os.listdir(class_dir) if f.endswith('.jpg')]
            if not slice_files:
                continue
            
            # 检查5个样本
            sample_size = min(5, len(slice_files))
            for slice_file in slice_files[:sample_size]:
                checked_slices += 1
                wsi_source, is_augmented, _ = get_wsi_source_from_slice_fixed(slice_file)
                
                if not wsi_source:
                    mapping_issues += 1
                    issues.append(f"无法解析WSI来源: {slice_file}")
    
    if mapping_issues == 0:
        print(f"✅ 切片-WSI映射正确 (检查了{checked_slices}个样本)")
    else:
        print(f"❌ 发现{mapping_issues}个映射问题")
    
    # ==================== 5. 生成fold检验报告 ====================
    print(f"\n{'='*60}")
    print(f"📋 Fold {fold_num} 检验报告")
    print(f"{'='*60}")
    
    # 汇总统计
    print(f"\n📊 汇总统计:")
    print(f"  总切片数: {stats['train_slices'] + stats['val_slices']}")
    print(f"  训练集/验证集比例: {stats['train_slices']/(stats['val_slices']+1e-6):.1f}:1")
    print(f"  增强数据占比: {stats['augmented_slices']['train']/(stats['train_slices']+1e-6)*100:.1f}%")
    print(f"  训练集WSI数: {len(stats['train_wsi_sources'])}")
    print(f"  验证集WSI数: {len(stats['val_wsi_sources'])}")
    
    if not issues:
        print(f"\n🎉 Fold {fold_num} 检验通过!")
        print(f"✅ 数据集质量良好，可以用于训练")
    else:
        print(f"\n⚠️  发现 {len(issues)} 个问题:")
        for i, issue in enumerate(issues[:10], 1):
            print(f"  {i}. {issue}")
        if len(issues) > 10:
            print(f"  ... 还有{len(issues)-10}个问题未显示")
    
    return issues, stats

# ==================== 主函数：检验所有fold ====================
def check_all_folds_fixed():
    """修复版：独立检验所有fold"""
    print("🔍 开始按fold独立检验数据集（修复版）")
    print("="*80)
    
    all_fold_issues = []
    all_fold_stats = []
    
    # 检查每个fold
    for fold_num in range(1, 5):
        issues, stats = check_single_fold_fixed(fold_num)
        all_fold_issues.append((fold_num, issues))
        all_fold_stats.append((fold_num, stats))
    
    # ==================== 生成总体验收报告 ====================
    print("\n" + "="*80)
    print("📊 总体验收报告（修复版）")
    print("="*80)
    
    total_issues = sum(len(issues) for _, issues in all_fold_issues)
    total_slices = sum(stats['train_slices'] + stats['val_slices'] for _, stats in all_fold_stats)
    
    print(f"\n📈 总体统计:")
    print(f"  总fold数: 4")
    print(f"  总切片数: {total_slices}")
    print(f"  总问题数: {total_issues}")
    
    # 各fold问题统计
    print(f"\n🔍 各fold问题统计:")
    for fold_num, issues in all_fold_issues:
        status = "✅ 通过" if len(issues) == 0 else f"⚠️  {len(issues)}个问题"
        print(f"  Fold {fold_num}: {status}")
    
    # WSI层面总体均衡性
    print(f"\n📊 WSI层面总体均衡性:")
    all_wsi_classes = defaultdict(int)
    
    for fold_num, stats in all_fold_stats:
        for wsi_source in stats['train_wsi_sources']:
            if wsi_source.startswith('AUG_'):
                aug_class, _, _ = parse_augmented_wsi_name(wsi_source)
                if aug_class:
                    all_wsi_classes[aug_class] += 1
            elif wsi_source in slide_labels:
                all_wsi_classes[slide_labels[wsi_source]] += 1
    
    for cls in ['Normal', 'Benign', 'In_Situ', 'Invasive']:
        count = all_wsi_classes.get(cls, 0)
        print(f"  {cls}: {count}个WSI")
    
    # 问题分类
    if total_issues > 0:
        print(f"\n📋 问题分类:")
        issue_categories = defaultdict(int)
        
        for fold_num, issues in all_fold_issues:
            for issue in issues:
                if '数据泄露' in issue or '重叠' in issue:
                    issue_categories['数据泄露'] += 1
                elif '增强' in issue or 'AUG' in issue or '元数据' in issue:
                    issue_categories['增强数据'] += 1
                elif '类别' in issue or 'Class' in issue or '不匹配' in issue:
                    issue_categories['类别问题'] += 1
                elif '映射' in issue or '解析' in issue:
                    issue_categories['映射问题'] += 1
                elif '均衡' in issue:
                    issue_categories['均衡性问题'] += 1
                else:
                    issue_categories['其他'] += 1
        
        for category, count in issue_categories.items():
            if count > 0:
                print(f"  {category}: {count}个")
        
        # 修复建议
        print(f"\n🔧 修复建议:")
        if issue_categories['数据泄露'] > 0:
            print("  1. 检查每个fold的训练/验证集划分，确保无WSI重叠")
        if issue_categories['增强数据'] > 0:
            print("  2. 生成缺失的增强WSI元数据文件")
            print("  3. 确保验证集不包含增强数据")
        if issue_categories['类别问题'] > 0:
            print("  4. 检查类别目录与文件类别的匹配关系")
        if issue_categories['均衡性问题'] > 0:
            print("  5. 注意：WSI分类任务关注WSI层面均衡，切片数量不均正常")
    else:
        print(f"\n🎉 所有fold检验通过！")
        print(f"✅ 数据集质量优秀，完全符合4-fold独立使用的要求")
        print(f"✅ 每个fold都是独立且自包含的数据集")
        print(f"✅ WSI层面均衡性良好，切片数量差异正常")
    
    print(f"\n{'='*80}")
    print("💡 重要说明：")
    print("  • WSI分类任务关注WSI层面的均衡，不是切片层面的均衡")
    print("  • 不同WSI的组织含量不同，切片数量自然不同")
    print("  • Invasive类别切片多是因为该类WSI组织更丰富")
    print("  • 这是正常现象，不是数据质量问题")
    print(f"{'='*80}")

# ==================== 执行检验 ====================
if __name__ == "__main__":
    check_all_folds_fixed()

🔍 开始按fold独立检验数据集质量（修复版）...
🔍 开始按fold独立检验数据集（修复版）

🔍 检验 Fold 1（修复版）

1️⃣ 数据泄露检验 (本fold内)
----------------------------------------
✅ 本fold内无数据泄露

2️⃣ 增强数据检验（修复版）
----------------------------------------
✅ 所有增强WSI都有元数据
✅ 训练集包含增强切片: 171个
✅ 验证集无增强数据 (正确)

3️⃣ 数据集均衡性检验（WSI层面）
----------------------------------------
📊 切片数量统计:
  训练集: 72000个切片
  验证集: 24000个切片

📊 WSI来源统计（这才是关键！）:
  训练集WSI: 24个
  验证集WSI: 8个
  其中增强WSI: 10个

📊 WSI层面类别分布（训练集）:
  Normal: 6个WSI
  Benign: 6个WSI
  In_Situ: 6个WSI
  Invasive: 6个WSI

📊 切片层面类别分布（训练集，仅供参考）:
  Normal: 18000个切片
  Benign: 18000个切片
  In_Situ: 18000个切片
  Invasive: 18000个切片

📈 WSI层面均衡性指标:
  每类平均WSI数: 6.0
  标准差: 0.0
  变异系数(CV): 0.000
  ✅ WSI层面均衡性良好

4️⃣ 切片-WSI映射检验
----------------------------------------
✅ 切片-WSI映射正确 (检查了20个样本)

📋 Fold 1 检验报告

📊 汇总统计:
  总切片数: 96000
  训练集/验证集比例: 3.0:1
  增强数据占比: 0.2%
  训练集WSI数: 24
  验证集WSI数: 8

🎉 Fold 1 检验通过!
✅ 数据集质量良好，可以用于训练

🔍 检验 Fold 2（修复版）

1️⃣ 数据泄露检验 (本fold内)
----------------------------------------
✅ 本fold内无数据泄露

2️⃣ 增强数据检验（修复